In [2]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

import sys
from pathlib import Path

import config
args = config.args

from helper import create_smart_stock_patterns, analyze_stock_mentions_fast, create_sequences, count_zeros_in_sequence

In [5]:
#complete comment data from 2019 - 2024, from r/wallstreetbets; 
df_comments = pd.read_csv(args.submissionandcomments_dir)
df_comments

C:\Users\Administrator\AppData\Local\Temp\ipykernel_13460\279796623.py:2: DtypeWarning: Columns (6,7,8,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df_comments = pd.read_csv(args.submissionandcomments_dir)


,Unnamed: 0,id,author,created_utc,score,subreddit,body,title,selftext,num_comments,url,upvote_ratio,combined_text,date
0,48897327,j2fxon7,ninkorn,1.672531e+09,2,wallstreetbets,"LOL, actually Sbarros is NYC tradition in a way",NaN,NaN,NaN,NaN,NaN,"LOL, actually Sbarros is NYC tradition in a way",2023-01-01 00:00:04
1,48897328,j2fxp7c,spellbadgrammargood,1.672531e+09,11,wallstreetbets,"STARTING TUESDAY I WILL MAKE $1,000 INTO A MIL...",NaN,NaN,NaN,NaN,NaN,"STARTING TUESDAY I WILL MAKE $1,000 INTO A M...",2023-01-01 00:00:12
2,48897329,j2fxp7s,whicky1978,1.672531e+09,2,wallstreetbets,Sounds about right. Oil does better in a reces...,NaN,NaN,NaN,NaN,NaN,Sounds about right. Oil does better in a rec...,2023-01-01 00:00:12
3,48897330,j2fxp85,Odd-Ask-139,1.672531e+09,2,wallstreetbets,from the pictures it looks like the best place...,NaN,NaN,NaN,NaN,NaN,from the pictures it looks like the best pla...,2023-01-01 00:00:12
4,48897331,j2fxqje,[deleted],1.672531e+09,4,wallstreetbets,[deleted],NaN,NaN,NaN,NaN,NaN,[deleted],2023-01-01 00:00:28
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11833065,61224148,1hqr466,Ok_Interaction5406,1.735689e+09,1,wallstreetbets,NaN,Stock Market the last three days of 2024,NaN,0.0,https://i.redd.it/uenz7owwu9ae1.png,1.0,Stock Market the last three days of 2024,2024-12-31 23:52:10
11833066,61224149,1hqr53z,WeekendConsistent547,1.735689e+09,1,wallstreetbets,NaN,HI ANYONE WHO RECOMMENED ANY GOOD COIN OR STOC...,[removed],1.0,https://www.reddit.com/r/wallstreetbets/commen...,1.0,[removed] HI ANYONE WHO RECOMMENED ANY GOOD CO...,2024-12-31 23:53:38
11833067,61224150,1hqr79i,Joey164,1.735689e+09,1,wallstreetbets,NaN,Not a bad year! Hopefully 2025 is better!,[removed],0.0,https://i.redd.it/fgjvekpsv9ae1.jpeg,1.0,[removed] Not a bad year! Hopefully 2025 is b...,2024-12-31 23:57:06
11833068,61224151,1hqr83p,Hot-Boat1925,1.735690e+09,1,wallstreetbets,NaN,Onds short sellers cost me 15k out of 60k in 2...,[removed],1.0,https://i.redd.it/hp7et460w9ae1.jpeg,1.0,[removed] Onds short sellers cost me 15k out o...,2024-12-31 23:58:29


In [6]:
#s&p stock information
df_stocks = pd.read_csv(args.sp_stocks_dir)
df_stocks.dropna(inplace=True)
df_stocks

,cik_str,ticker,title,name
0,1045810,NVDA,NVIDIA CORP,NVIDIA
1,789019,MSFT,MICROSOFT CORP,MICROSOFT
2,320193,AAPL,Apple Inc.,Apple.
3,1652044,GOOGL,Alphabet Inc.,Alphabet.
4,1018724,AMZN,AMAZON COM INC,AMAZONCOM
...,...,...,...,...
456,58492,LEG,LEGGETT & PLATT INC,LEGGETT & PLATT
457,1652044,GOOG,Alphabet Inc.,Alphabet.
458,1754301,FOX,Fox Corp,Fox
459,1564708,NWS,NEWS CORP,NEWS


In [7]:
#1. 
#(1) pair submission or comment with stock tickers
stock_patterns = create_smart_stock_patterns(df_stocks)
mentions_df = analyze_stock_mentions_fast(df_comments, stock_patterns, batch_size=10000)
mentions_df['text_date'] = pd.to_datetime(mentions_df['created_utc'], unit='s', utc = True).dt.date
mentions_df['combined_text'] = (
    mentions_df['title'].fillna('') + ' ' + 
    mentions_df['selftext'].fillna('') + ' ' + 
    mentions_df['body'].fillna('')
).str.strip()

Processing batch 1/1184
Processing batch 2/1184
Processing batch 3/1184
Processing batch 4/1184
Processing batch 5/1184
Processing batch 6/1184
Processing batch 7/1184
Processing batch 8/1184
Processing batch 9/1184
Processing batch 10/1184
Processing batch 11/1184
Processing batch 12/1184
Processing batch 13/1184
Processing batch 14/1184
Processing batch 15/1184
Processing batch 16/1184
Processing batch 17/1184
Processing batch 18/1184
Processing batch 19/1184
Processing batch 20/1184
Processing batch 21/1184
Processing batch 22/1184
Processing batch 23/1184
Processing batch 24/1184
Processing batch 25/1184
Processing batch 26/1184
Processing batch 27/1184
Processing batch 28/1184
Processing batch 29/1184
Processing batch 30/1184
Processing batch 31/1184
Processing batch 32/1184
Processing batch 33/1184
Processing batch 34/1184
Processing batch 35/1184
Processing batch 36/1184
Processing batch 37/1184
Processing batch 38/1184
Processing batch 39/1184
Processing batch 40/1184
Processin

In [ ]:
#(2) aggregate text by date
results_df = mentions_df.groupby(['ticker', 'text_date']).agg({
    'combined_text': lambda x: ' '.join(x), 
    'id': 'count' ,
    'score': 'sum'
}).reset_index()

results_df['text_date'] = pd.to_datetime(results_df['text_date'])
results_df.columns = ['ticker', 'date', 'combined_text', 'num_posts', 'num_of_net_upvotes']

In [ ]:
#2. create sequence for embedding
#earnings surprise information, including: fiscal end date, reported date, reported eps, estimated eps and surprise percent
earnings = pd.read_csv(args.eps_surprise_dir) 
earnings['reddit_sequence'] = earnings.apply(
    lambda row: create_sequences(row, results_df), axis=1
)

# earnings dataframe now has:
# ticker | earnings_date | surprise | reddit_sequence
# TSLA   | 2020-02-15    | 0.05     | [day1_text, day2_text, ..., day60_text]
# AAPL   | 2020-02-20    | -0.02    | [day1_text, day2_text, ..., day60_text]

In [ ]:
#3. finBert for embedding
from transformers import AutoTokenizer, AutoModel
import torch

tokenizer = AutoTokenizer.from_pretrained('ProsusAI/finbert')
model = AutoModel.from_pretrained('ProsusAI/finbert')

def get_bert_embedding(text):
    if not text or text.strip() == '':  # handle empty days
        return torch.zeros(768)
    
    inputs = tokenizer(text, return_tensors='pt', 
                      truncation=True, max_length=512,
                      padding=True)
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Use [CLS] token embedding
    embedding = outputs.last_hidden_state[:, 0, :].squeeze()
    return embedding

# Apply to each day in each sequence
def embed_sequence(text_sequence):
    embeddings = [get_bert_embedding(text) for text in text_sequence]
    return torch.stack(embeddings) 

earnings['embedding_sequence'] = earnings['reddit_sequence'].apply(embed_sequence)
# ticker | earnings_date | surprise | embedding_sequence
# TSLA   | 2020-02-15    | +5%      | Tensor(60, 768)
# AAPL   | 2020-02-20    | -2%      | Tensor(60, 768)

In [ ]:
#visualizing embedding results

import matplotlib.pyplot as plt
print("=== ANALYZING DAILY NUMBER OF POSTS IN THE LOOK-BACK WINDOW ===\n")
earnings['zero_post_days'] = earnings['embedding_sequence'].apply(count_zeros_in_sequence)
earnings['non_zero_post_days'] = 60 - earnings['zero_days']

print("Zero days distribution:")
print(earnings['zero_days'].describe())

print("\n=== Breakdown ===")
print(f"Samples with 0 posts (all zeros):     {(earnings['zero_days'] == 60).sum():4d} ({(earnings['zero_days'] == 60).sum()/len(earnings)*100:.1f}%)")
print(f"Samples with 1-10 posts:              {((earnings['non_zero_days'] >= 1) & (earnings['non_zero_days'] <= 10)).sum():4d} ({((earnings['non_zero_days'] >= 1) & (earnings['non_zero_days'] <= 10)).sum()/len(earnings)*100:.1f}%)")
print(f"Samples with 11-20 posts:             {((earnings['non_zero_days'] >= 11) & (earnings['non_zero_days'] <= 20)).sum():4d} ({((earnings['non_zero_days'] >= 11) & (earnings['non_zero_days'] <= 20)).sum()/len(earnings)*100:.1f}%)")
print(f"Samples with 21-40 posts:             {((earnings['non_zero_days'] >= 21) & (earnings['non_zero_days'] <= 40)).sum():4d} ({((earnings['non_zero_days'] >= 21) & (earnings['non_zero_days'] <= 40)).sum()/len(earnings)*100:.1f}%)")
print(f"Samples with 41-60 posts (very active): {(earnings['non_zero_days'] >= 41).sum():4d} ({(earnings['non_zero_days'] >= 41).sum()/len(earnings)*100:.1f}%)")

print(f"\nTotal samples: {len(earnings)}")
print(f"Mean posts per sample: {earnings['non_zero_days'].mean():.1f} days")
print(f"Median posts per sample: {earnings['non_zero_days'].median():.1f} days")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(earnings['non_zero_days'], bins=30, edgecolor='black', alpha=0.7)
axes[0].axvline(earnings['non_zero_days'].mean(), color='red', linestyle='--', label=f'Mean: {earnings["non_zero_days"].mean():.1f}')
axes[0].set_xlabel('Number of Days with Posts')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Post Activity')
axes[0].legend()
axes[0].grid(alpha=0.3)

# By class
pos_mask = earnings['surprise'] > 0
axes[1].hist([earnings[pos_mask]['non_zero_days'], earnings[~pos_mask]['non_zero_days']], 
             bins=30, label=['Positive Surprise', 'Negative Surprise'], alpha=0.7)
axes[1].set_xlabel('Number of Days with Posts')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Post Activity by Surprise Direction')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
earnings = earnings.sort_values('earnings_date').reset_index(drop=True)

# Define your date splits 
train_end_date = pd.to_datetime('2024-06-30') #the current date value is only a showcase
val_end_date = pd.to_datetime('2024-09-30')

# Create boolean masks based on dates
train_mask = earnings['earnings_date'] <= train_end_date
val_mask = (earnings['earnings_date'] > train_end_date) & (earnings['earnings_date'] <= val_end_date)
test_mask = earnings['earnings_date'] > val_end_date

# Split data using date-based masks
train_earnings = earnings[train_mask]
val_earnings = earnings[val_mask]
test_earnings = earnings[test_mask]

# Stack embeddings into tensors
X_train = torch.stack(train_earnings['embedding_sequence'].tolist())
y_train = torch.tensor((train_earnings['surprise'] > 0).values, dtype=torch.float32)

X_val = torch.stack(val_earnings['embedding_sequence'].tolist())
y_val = torch.tensor((val_earnings['surprise'] > 0).values, dtype=torch.float32)

X_test = torch.stack(test_earnings['embedding_sequence'].tolist())
y_test = torch.tensor((test_earnings['surprise'] > 0).values, dtype=torch.float32)

save_path = args.dataset_save_dir
dir = Path(save_path)
dir.mkdir(parents=True, exist_ok=True)

np.save(save_path / 'X_train.npy', X_train.cpu().numpy())
print(f"✓ Saved X_train.npy (shape: {X_train.shape})")
np.save(save_path / 'y_train.npy', y_train.cpu().numpy())
print(f"✓ Saved y_train.npy (shape: {y_train.shape})")
np.save(save_path / 'X_val.npy', X_val.cpu().numpy())
print(f"✓ Saved X_val.npy (shape: {X_val.shape})")
np.save(save_path / 'y_val.npy', y_val.cpu().numpy())
print(f"✓ Saved y_val.npy (shape: {y_val.shape})")
np.save(save_path / 'X_test.npy', X_test.cpu().numpy())
print(f"✓ Saved X_test.npy (shape: {X_test.shape})")
np.save(save_path / 'y_test.npy', y_test.cpu().numpy())
print(f"✓ Saved y_test.npy (shape: {y_test.shape})")
print("\nAll files saved successfully!")